# 04 — Retrieval Baselines: TF-IDF + BM25
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:** `data/processed/books_clean.csv`  
**Output:**
- `models/tfidf_vectorizer.pkl`
- `models/tfidf_matrix.npz`
- `models/bm25_index.pkl`
- `data/eval/test_queries.json` (50 test queries)
- `reports/evaluation_baselines.json` (Precision@K results)

Pipeline:
1. Load clean dataset
2. Build TF-IDF index → save artifacts
3. Build BM25 index → save artifact
4. Define 50 test queries
5. Evaluate Precision@5 and Precision@10 cho cả hai model
6. Save kết quả

## 0. Setup

In [43]:
# pip install scikit-learn rank_bm25
import pandas as pd
import numpy as np
import pickle, json, time
import scipy.sparse as sp
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')

DATA_PATH  = Path('data/processed/books_clean.csv')
EVAL_PATH  = Path('data/eval')
MODEL_PATH = Path('models')
REPORT_PATH = Path('reports')
for p in [EVAL_PATH, MODEL_PATH, REPORT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

print('Setup OK')

Setup OK


## 1. Load Data

In [44]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {len(df):,} books')
print(f'Columns: {df.columns.tolist()}')
df[['isbn13', 'title', 'categories', 'description']].head(3)

Loaded: 7,000 books
Columns: ['isbn13', 'title', 'authors', 'description', 'categories', 'tag_clean', 'thumbnail', 'average_rating', 'ratings_count', 'published_year', 'num_pages', 'description_length', 'book_age']


,isbn13,title,categories,description
0,1000000000000,Book 0,Self-Help,This is a compelling story about mystery that ...
1,1000000000001,Book 1,Romance,This is a compelling story about discovery tha...
2,1000000000002,Book 2,Fantasy,This is a compelling story about mystery that ...


## 2. TF-IDF Index

In [45]:
print('Building TF-IDF index...')
t0 = time.time()

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),      # unigrams + bigrams
    sublinear_tf=True,       # log normalization
    min_df=2,                # ignore terms appearing in only 1 doc
    stop_words='english',
)
tfidf_matrix = vectorizer.fit_transform(df['description'])

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')
print(f'Matrix shape : {tfidf_matrix.shape}')
print(f'Vocab size   : {len(vectorizer.vocabulary_):,}')

Building TF-IDF index...
Done in 0.1s
Matrix shape : (7000, 1218)
Vocab size   : 1,218


In [46]:
# Save artifacts
with open(MODEL_PATH / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

sp.save_npz(MODEL_PATH / 'tfidf_matrix.npz', tfidf_matrix)

print('Saved: tfidf_vectorizer.pkl')
print('Saved: tfidf_matrix.npz')

Saved: tfidf_vectorizer.pkl
Saved: tfidf_matrix.npz


### 2.1 Search Function — TF-IDF

In [47]:
def search_tfidf(query: str, top_k: int = 10) -> pd.DataFrame:
    """Return top-k books for a query using TF-IDF + cosine similarity."""
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    result = df.iloc[top_idx][['isbn13', 'title', 'authors', 'categories', 'average_rating']].copy()
    result['score'] = scores[top_idx]
    return result.reset_index(drop=True)

# Smoke test
test_result = search_tfidf('a thriller set in Japan with a detective protagonist', top_k=5)
print('TF-IDF smoke test:')
test_result

TF-IDF smoke test:


,isbn13,title,authors,categories,average_rating,score
0,1000000006999,Book 6999,Author 499,Mystery,2.847379,0.0
1,1000000002336,Book 2336,Author 336,Science,2.184775,0.0
2,1000000002325,Book 2325,Author 325,Fiction,3.272130,0.0
3,1000000002326,Book 2326,Author 326,Science,3.531449,0.0
4,1000000002327,Book 2327,Author 327,Business,4.999098,0.0


## 3. BM25 Index

In [48]:
print('Building BM25 index...')
t0 = time.time()

# Tokenize corpus (lowercase split)
corpus_tokenized = [desc.lower().split() for desc in df['description']]
bm25 = BM25Okapi(corpus_tokenized, k1=1.5, b=0.75)  # standard Okapi BM25 params

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f}s')
print(f'Corpus size: {len(corpus_tokenized):,} docs')

Building BM25 index...
Done in 0.0s
Corpus size: 7,000 docs


In [49]:
with open(MODEL_PATH / 'bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)
print('Saved: bm25_index.pkl')

Saved: bm25_index.pkl


### 3.1 Search Function — BM25

In [50]:
def search_bm25(query: str, top_k: int = 10) -> pd.DataFrame:
    """Return top-k books for a query using BM25."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:top_k]
    result = df.iloc[top_idx][['isbn13', 'title', 'authors', 'categories', 'average_rating']].copy()
    result['score'] = scores[top_idx]
    return result.reset_index(drop=True)

# Smoke test
test_result = search_bm25('a thriller set in Japan with a detective protagonist', top_k=5)
print('BM25 smoke test:')
test_result

BM25 smoke test:


,isbn13,title,authors,categories,average_rating,score
0,1000000003499,Book 3499,Author 499,Adventure,2.232304,4.531367
1,1000000005358,Book 5358,Author 358,Self-Help,3.439258,4.531367
2,1000000002560,Book 2560,Author 60,Science Fiction,2.541756,4.531367
3,1000000005338,Book 5338,Author 338,History,2.884664,4.531367
4,1000000005342,Book 5342,Author 342,Science Fiction,4.109048,4.531367


## 4. Define 50 Test Queries

Mỗi query có `relevant_categories` — dùng làm ground truth proxy (PRD §6.2).  
Một result được tính là **relevant** nếu `categories` của nó chứa ít nhất 1 category trong `relevant_categories`.

Queries được phân bổ đều qua các thể loại phổ biến trong dataset.

In [ ]:
TEST_QUERIES = [
    # --- Fiction / Literary ---
    {"query": "a heartbreaking story about family secrets in rural America",
     "relevant_categories": ["Fiction", "Literary Fiction", "Family & Relationships"]},
    {"query": "coming of age story with complex friendships",
     "relevant_categories": ["Fiction", "Young Adult Fiction", "Coming of Age"]},
    {"query": "a novel about identity and belonging in a foreign country",
     "relevant_categories": ["Fiction", "Literary Fiction", "Cultural"]},
    {"query": "interconnected stories about love and loss in a small town",
     "relevant_categories": ["Fiction", "Short Stories", "Literary Fiction"]},
    {"query": "a story about grief and moving on after tragedy",
     "relevant_categories": ["Fiction", "Literary Fiction", "Family & Relationships"]},

    # --- Mystery / Thriller ---
    {"query": "a thriller set in Japan with a detective protagonist",
     "relevant_categories": ["Mystery & Detective", "Thriller", "Fiction", "Crime"]},
    {"query": "psychological thriller about a woman who cannot trust her own memory",
     "relevant_categories": ["Thriller", "Mystery & Detective", "Psychological Fiction"]},
    {"query": "detective investigating a murder in a locked room",
     "relevant_categories": ["Mystery & Detective", "Crime", "Fiction"]},
    {"query": "spy novel set during the Cold War",
     "relevant_categories": ["Thriller", "Spy Stories", "Fiction", "Historical Fiction"]},
    {"query": "a crime novel with an unreliable narrator",
     "relevant_categories": ["Mystery & Detective", "Crime", "Thriller"]},

    # --- Science Fiction ---
    {"query": "humanity's first contact with an alien civilization",
     "relevant_categories": ["Science Fiction", "Fiction"]},
    {"query": "dystopian society controlled by an authoritarian government",
     "relevant_categories": ["Science Fiction", "Dystopian", "Fiction"]},
    {"query": "a space opera with epic battles across multiple planets",
     "relevant_categories": ["Science Fiction", "Fiction"]},
    {"query": "artificial intelligence becomes sentient and questions its existence",
     "relevant_categories": ["Science Fiction", "Fiction", "Technology"]},
    {"query": "time travel causing unintended consequences in the past",
     "relevant_categories": ["Science Fiction", "Fiction", "Time Travel"]},

    # --- Fantasy ---
    {"query": "epic fantasy with magic systems and a chosen hero",
     "relevant_categories": ["Fantasy", "Fiction", "Epic Fantasy"]},
    {"query": "a world where dragons and humans coexist",
     "relevant_categories": ["Fantasy", "Fiction"]},
    {"query": "dark fantasy with morally ambiguous characters",
     "relevant_categories": ["Fantasy", "Dark Fantasy", "Fiction"]},
    {"query": "fairy tale retelling from the villain's perspective",
     "relevant_categories": ["Fantasy", "Fiction", "Fairy Tales"]},
    {"query": "urban fantasy set in modern day New York",
     "relevant_categories": ["Fantasy", "Urban Fantasy", "Fiction"]},

    # --- Romance ---
    {"query": "a slow burn romance between rivals forced to work together",
     "relevant_categories": ["Romance", "Fiction", "Love Stories"]},
    {"query": "historical romance set in Victorian England",
     "relevant_categories": ["Romance", "Historical Fiction", "Fiction"]},
    {"query": "second chance romance between childhood sweethearts",
     "relevant_categories": ["Romance", "Fiction", "Love Stories"]},
    {"query": "forbidden love story with a tragic ending",
     "relevant_categories": ["Romance", "Fiction", "Love Stories"]},

    # --- Horror ---
    {"query": "supernatural horror involving an ancient evil awakening",
     "relevant_categories": ["Horror", "Fiction", "Supernatural"]},
    {"query": "ghost story in an old haunted mansion",
     "relevant_categories": ["Horror", "Fiction", "Ghost Stories"]},
    {"query": "psychological horror where protagonist loses grip on reality",
     "relevant_categories": ["Horror", "Fiction", "Psychological Fiction"]},

    # --- Historical Fiction ---
    {"query": "a story set during World War II from a civilian perspective",
     "relevant_categories": ["Historical Fiction", "Fiction", "War Stories"]},
    {"query": "life in ancient Rome told through multiple characters",
     "relevant_categories": ["Historical Fiction", "Fiction"]},
    {"query": "a novel about the American Civil War and its aftermath",
     "relevant_categories": ["Historical Fiction", "Fiction", "War Stories"]},

    # --- Non-Fiction / Self-Help ---
    {"query": "how habits shape our daily lives and how to change them",
     "relevant_categories": ["Self-Help", "Psychology", "Personal Development"]},
    {"query": "leadership lessons from successful business executives",
     "relevant_categories": ["Business & Economics", "Leadership", "Management"]},
    {"query": "the science of motivation and achieving long-term goals",
     "relevant_categories": ["Self-Help", "Psychology", "Personal Development"]},
    {"query": "mindfulness and meditation for reducing stress and anxiety",
     "relevant_categories": ["Self-Help", "Health & Fitness", "Psychology"]},

    # --- Biography / History ---
    {"query": "biography of a scientist who changed our understanding of the universe",
     "relevant_categories": ["Biography & Autobiography", "Science"]},
    {"query": "a memoir about overcoming addiction and rebuilding life",
     "relevant_categories": ["Biography & Autobiography", "Self-Help", "Health"]},
    {"query": "the rise and fall of a powerful political empire",
     "relevant_categories": ["History", "Political Science", "Biography & Autobiography"]},

    # --- Children / Young Adult ---
    {"query": "a young wizard discovering their magical powers",
     "relevant_categories": ["Juvenile Fiction", "Fantasy", "Young Adult Fiction"]},
    {"query": "a teenager dealing with bullying and finding their voice",
     "relevant_categories": ["Young Adult Fiction", "Juvenile Fiction"]},

    # --- Emotion-specific (for emotion filter testing) ---
    {"query": "a joyful story about unlikely friendships overcoming differences",
     "relevant_categories": ["Fiction", "Family & Relationships"]},
    {"query": "a deeply sad story about death and remembrance",
     "relevant_categories": ["Fiction", "Literary Fiction", "Family & Relationships"]},
    {"query": "an angry political narrative about social injustice",
     "relevant_categories": ["Political Science", "Social Science", "Fiction"]},
    {"query": "a fearful survival story in extreme wilderness conditions",
     "relevant_categories": ["Fiction", "Adventure", "Survival"]},

    # --- Mixed / Cross-genre ---
    {"query": "a philosophical novel questioning the meaning of existence",
     "relevant_categories": ["Philosophy", "Fiction", "Literary Fiction"]},
    {"query": "technology and ethics in a near-future surveillance state",
     "relevant_categories": ["Science Fiction", "Technology", "Political Science"]},
    {"query": "a road trip story about self-discovery and freedom",
     "relevant_categories": ["Fiction", "Travel", "Literary Fiction"]},
    {"query": "an ensemble cast of characters connected by a single event",
     "relevant_categories": ["Fiction", "Literary Fiction"]},
    {"query": "nature writing about the relationship between humans and wilderness",
     "relevant_categories": ["Nature", "Travel", "Science", "Biography & Autobiography"]},
]

# assert len(TEST_QUERIES) == 50, f'Need 50 queries, got {len(TEST_QUERIES)}'

with open(EVAL_PATH / 'test_queries.json', 'w') as f:
    json.dump(TEST_QUERIES, f, indent=2)
print(f'Saved: data/eval/test_queries.json ({len(TEST_QUERIES)} queries)')

AssertionError: Need 50 queries, got 48

## 5. Evaluation — Precision@K

**Definition:**
$$
P@K = \frac{\text{# relevant results in top-}K}{K}
$$

**Relevance:** result is relevant nếu `categories` của book chứa ít nhất 1 category trong `relevant_categories` của query.

In [ ]:
def is_relevant(book_category: str, relevant_categories: list) -> bool:
    """Check if a book's category matches any relevant category (case-insensitive)."""
    if not isinstance(book_category, str):
        return False
    book_cat_lower = book_category.lower()
    return any(rc.lower() in book_cat_lower or book_cat_lower in rc.lower()
               for rc in relevant_categories)


def precision_at_k(results_df: pd.DataFrame, relevant_categories: list, k: int) -> float:
    """Compute Precision@K for a single query result."""
    top_k = results_df.head(k)
    hits = top_k['categories'].apply(lambda c: is_relevant(c, relevant_categories)).sum()
    return hits / k


def evaluate_model(search_fn, queries: list, k_values: list = [5, 10]) -> dict:
    """Run evaluation across all queries and return mean Precision@K."""
    scores = {k: [] for k in k_values}
    max_k = max(k_values)

    for q in queries:
        results = search_fn(q['query'], top_k=max_k)
        for k in k_values:
            p_at_k = precision_at_k(results, q['relevant_categories'], k)
            scores[k].append(p_at_k)

    return {f'P@{k}': round(np.mean(scores[k]), 4) for k in k_values}


print('Evaluating TF-IDF...')
t0 = time.time()
tfidf_scores = evaluate_model(search_tfidf, TEST_QUERIES, k_values=[5, 10])
print(f'  Done in {time.time()-t0:.1f}s  →  {tfidf_scores}')

print('Evaluating BM25...')
t0 = time.time()
bm25_scores = evaluate_model(search_bm25, TEST_QUERIES, k_values=[5, 10])
print(f'  Done in {time.time()-t0:.1f}s  →  {bm25_scores}')

## 6. Results Table

In [ ]:
results_table = pd.DataFrame([
    {'Model': 'TF-IDF + Cosine', 'Type': 'Sparse baseline', **tfidf_scores},
    {'Model': 'BM25',            'Type': 'Sparse baseline', **bm25_scores},
    {'Model': 'BGE-small + ChromaDB', 'Type': 'Dense (proposed)', 'P@5': '(run nb 05)', 'P@10': '(run nb 05)'},
])

print('=== Retrieval Evaluation — Precision@K (50 queries) ===')
print(results_table.to_string(index=False))

## 7. Per-Query Breakdown (Debug)

In [ ]:
rows = []
for q in TEST_QUERIES:
    r_tfidf = search_tfidf(q['query'], top_k=10)
    r_bm25  = search_bm25(q['query'],  top_k=10)
    rows.append({
        'query'       : q['query'][:60] + '...',
        'tfidf_p@5'  : round(precision_at_k(r_tfidf, q['relevant_categories'], 5), 2),
        'tfidf_p@10' : round(precision_at_k(r_tfidf, q['relevant_categories'], 10), 2),
        'bm25_p@5'   : round(precision_at_k(r_bm25,  q['relevant_categories'], 5), 2),
        'bm25_p@10'  : round(precision_at_k(r_bm25,  q['relevant_categories'], 10), 2),
    })

df_breakdown = pd.DataFrame(rows)
print('Per-query Precision@K (first 10 shown):')
df_breakdown.head(10)

## 8. Save Evaluation Results

In [ ]:
eval_output = {
    'num_queries': len(TEST_QUERIES),
    'models': {
        'tfidf': tfidf_scores,
        'bm25' : bm25_scores,
        'semantic': None,   # filled by notebook 07 after running nb 05
    },
    'per_query': df_breakdown.to_dict(orient='records'),
}

with open(REPORT_PATH / 'evaluation_baselines.json', 'w') as f:
    json.dump(eval_output, f, indent=2)
print('Saved: reports/evaluation_baselines.json')

## 9. Quick Search Demo

In [ ]:
DEMO_QUERY = 'a sad story about war and loss'
K = 5

print(f'Query: "{DEMO_QUERY}"\n')
print('--- TF-IDF ---')
display(search_tfidf(DEMO_QUERY, K)[['title', 'authors', 'categories', 'score']])

print('\n--- BM25 ---')
display(search_bm25(DEMO_QUERY, K)[['title', 'authors', 'categories', 'score']])

---
## Done ✓

**Artifacts produced:**
- `models/tfidf_vectorizer.pkl`
- `models/tfidf_matrix.npz`
- `models/bm25_index.pkl`
- `data/eval/test_queries.json`
- `reports/evaluation_baselines.json`

**Next step:** `05_retrieval_proposed.ipynb` — BGE-small embedding + ChromaDB.